# 1. DC Motor Dynamics

**Build a motor model before building its controller.**

Imagine connecting a stationary motor to a battery. The shaft does not instantly jump to its final speed. Current builds, the motor produces torque, and the shaft accelerates. As it spins, it also generates a voltage that changes the current flowing through it.

Our goal is to explain that sequence and turn it into a simulation. You will write the code; this notebook provides the physics, worked examples, and implementation checkpoints.

By the end, you will have a function describing the motor's dynamics, a startup simulation, and plots checked against an independent calculation. No physical motor or prior control theory is required.

**Route through the lesson:** electrical energy → rotational motion → two coupled equations → numerical integration → plots and checks.

**Using the exercises:** try each code cell before opening **Show a hint** or **Show solution and explanation** below it. Solutions build on earlier exercises in order. They are reference text, so Run All does not execute them; copy a solution into your exercise cell if you want to run it.


[Next lesson: Open-Loop Control](02_open_loop_control.ipynb)

## 1.1 From electrical energy to rotation

<div style="text-align: center;">
<img src="assets/dc-motor-overview.svg" alt="End-on motor schematic with N and S poles, a rotor, opposite winding currents, a rightward magnetic field, and opposing forces that produce clockwise torque." style="display: block; margin-left: auto; margin-right: auto; width: 100%; max-width: 100%; height: auto;">
</div>

<div class="figure-caption" style="box-sizing: border-box; width: 100%; background-color: rgba(128, 128, 128, 0.10); padding: 12px 18px; margin: 0 0 20px;">

**Figure 1.** The dot indicates current toward you; the cross indicates current away from you. The magnetic field $B$ points from N to S. The two forces $F$ act on opposite sides of the rotor and together produce clockwise torque. Winding end connections and the commutator are omitted for clarity.

</div>

**Current** is the rate at which electric charge flows. We measure it in amperes (A). **Voltage** is a difference in electrical potential: energy available per unit charge, measured in volts (V). Applying voltage across a motor's windings can drive current through them.

A current-carrying wire in a magnetic field experiences a force. Inside a motor, those forces create a turning effect on the rotor, called **torque**, measured in newton-metres (N·m). A brushed DC motor uses its commutator to switch winding connections as it turns, keeping the torque acting in the intended direction. We'll model the average behavior without simulating that switching.

Torque and speed are different quantities. Torque is a twisting effort; speed describes existing motion. Net torque causes a **change** in speed, just as net force causes a change in linear velocity.

There is also an effect in the other direction: a rotating motor generates voltage. This is **back electromotive force**, or **back EMF**. Despite the name, EMF is a voltage, not a mechanical force. In normal motoring, it opposes the applied voltage, reducing the voltage available to drive current.

```text
Applied voltage → Current → Torque → Acceleration → Speed
                     ↑                                │
                     └──────── Back EMF ───────────────┘
```

This is a physical coupling inside the motor. We haven't added a feedback controller yet.

### Our modeling assumptions

We'll use an idealized brushed DC motor with a constant magnetic field. Winding resistance, inductance, inertia, and friction coefficients will remain fixed. We'll include friction proportional to speed, but omit static friction, brush voltage drops, heating, and magnetic saturation. The voltage source is ideal.

Choose one rotation direction as positive. Positive current produces positive torque. An external load torque that we subtract is positive when it acts against positive rotation. It is a signed input, not a friction law that automatically reverses with motion.

## 1.2 What changes, and what stays fixed?

Our **state** contains the two quantities needed to describe the motor's evolving electrical and mechanical condition: current $i$ and angular velocity $\omega$.

Angular velocity is measured in radians per second. One revolution is $2\pi$ radians, so:

$$
\text{speed in rpm}=\omega\frac{60}{2\pi}.
$$

For example, $10\ \mathrm{rad/s}$ is about $95.5\ \mathrm{rpm}$. We'll calculate in SI units and convert only for display.

| Role | Mathematical symbol | Python name | Meaning |
| --- | --- | --- | --- |
| State | $i$ | `current` | Winding current, in A |
| State | $\omega$ | `omega` | Angular velocity, in rad/s |
| Input | $V$ | `voltage` | Applied voltage, in V |
| Input | $\tau_{\mathrm{load}}$ | `load_torque` | External load torque, in N·m |
| Calculated output | $\tau_m$ | `motor_torque` | Torque produced by current, in N·m |
| Calculated output | $e_b$ | `back_emf` | Voltage generated by rotation, in V |

A **parameter** describes the motor rather than its momentary state. We'll use the following illustrative values; they are not specifications for a particular real motor.

| Symbol / Python name | Meaning | Value | Unit |
| --- | --- | --- | --- |
| $R$ / `R` | Winding resistance | 1.0 | Ω |
| $L$ / `L` | Winding inductance | 0.01 | H |
| $K_t$ / `K_t` | Torque per ampere | 0.1 | N·m/A |
| $K_e$ / `K_e` | Back EMF per unit speed | 0.1 | V·s/rad |
| $J$ / `J` | Rotational inertia | 0.01 | kg·m² |
| $b$ / `b` | Viscous friction coefficient | 0.001 | N·m·s/rad |

For this ideal motor, $K_t$ and $K_e$ have equal numerical values in these SI units. They describe different relationships, so we keep separate names.

We don't need shaft angle to simulate speed here: none of our equations depends on it. A position model would add angle as another state.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 1 — Set up the experiment

Define the six parameters with the values and units above. Individual variables are enough; you do not need a class yet.

Define the experiment inputs separately: $V=6\ \mathrm{V}$ and $\tau_{\mathrm{load}}=0\ \mathrm{N\cdot m}$. Start with zero current and zero speed. Use the state order **`[current, omega]`** throughout the notebook.

**Completion check:** you can identify which values describe the motor, which are inputs, and which will change during integration.

</div>


<details>
<summary>Show a hint</summary>

Separate the fixed motor parameters from the experiment inputs. The initial state contains current first, then angular velocity.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
import numpy as np

# Motor parameters (SI units)
R = 1.0       # ohm
L = 0.01      # H
K_t = 0.1     # N m / A
K_e = 0.1     # V s / rad
J = 0.01      # kg m^2
b = 0.001     # N m s / rad

# Experiment inputs and initial state
voltage = 6.0       # V
load_torque = 0.0   # N m
initial_state = [0.0, 0.0]  # [current (A), omega (rad/s)]
```

The six parameters describe the motor. Voltage and load describe this experiment. The solver will evolve the two state variables; it will not change the parameters.

</details>


## 1.3 Derive the electrical equation

<div style="text-align: center;">
<img src="assets/motor-electrical-model.svg" alt="Compact series circuit with supply V, resistance R, inductance L, back EMF e sub b, clockwise current, and reference polarities." style="display: block; margin-left: auto; margin-right: auto; width: 100%; max-width: 100%; height: auto;">
</div>

<div class="figure-caption" style="box-sizing: border-box; width: 100%; background-color: rgba(128, 128, 128, 0.10); padding: 12px 18px; margin: 0 0 20px;">

**Figure 2.** Follow positive current $i$ clockwise: the supply $V$ provides a voltage rise, while resistance $R$, inductance $L$, and back EMF $e_b$ account for it. The marked polarities define our voltage references; the inductor voltage can be negative when current decreases.

</div>

Three voltage contributions account for the voltage applied across our motor.

**Resistance.** The windings resist current and dissipate energy as heat. Ohm's law gives their resistive voltage drop:

$$V_R=Ri.$$

For a $1\ \Omega$ resistance carrying $2\ \mathrm{A}$, the drop is $2\ \mathrm{V}$.

**Inductance.** Current creates a magnetic field that stores energy. Changing that current requires voltage across the inductance:

$$V_L=L\frac{di}{dt}.$$

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Key distinction**

The derivative $di/dt$ means the instantaneous **rate of change of current**, in A/s. A value of $20\ \mathrm{A/s}$ means that, if this rate stayed constant for $0.01\ \mathrm{s}$, current would increase by $0.2\ \mathrm{A}$. It does not mean the current is 20 A.

</div>

Finite voltage across a nonzero inductance produces a finite rate of change. Current therefore cannot jump instantaneously in this model.

**Back EMF.** The generated voltage is proportional to angular velocity:

$$e_b=K_e\omega.$$

At $20\ \mathrm{rad/s}$, our motor generates $2\ \mathrm{V}$ of back EMF.

### Put the voltage balance together

Kirchhoff's voltage law says the signed voltage changes around a circuit loop sum to zero. For our motor:

$$
V=Ri+L\frac{di}{dt}+K_e\omega.
$$

To simulate current, isolate its rate of change:

$$
\boxed{\frac{di}{dt}=\frac{V-Ri-K_e\omega}{L}}.
$$

The numerator is the voltage left across the inductance. If it is positive, current increases. If negative, current decreases. If zero, current is momentarily unchanged—even if current itself is nonzero.

At the starting instant, $i=0$ and $\omega=0$, so:

$$
\left.\frac{di}{dt}\right|_{t=0}
=\frac{6}{0.01}=600\ \mathrm{A/s}.
$$

That is the **initial slope**, not the starting current. Current starts at zero. As it rises, the resistive drop grows; as the motor accelerates, back EMF grows too. The initial slope will not persist.

**Unit check:** a henry has units of V·s/A, so voltage divided by inductance has units A/s, exactly what a current derivative needs.

## 1.4 Derive the mechanical equation

<div style="text-align: center;">
<img src="assets/motor-mechanical-model.svg" alt="Rotor inertia J with clockwise motor torque and a single counterclockwise arrow for combined friction and external load torque." style="display: block; margin-left: auto; margin-right: auto; width: 100%; max-width: 100%; height: auto;">
</div>

<div class="figure-caption" style="box-sizing: border-box; width: 100%; background-color: rgba(128, 128, 128, 0.10); padding: 12px 18px; margin: 0 0 20px;">

**Figure 3.** Positive rotation is clockwise. Motor torque $\tau_m$ acts clockwise; for this illustrated case, viscous friction $b\omega$ and a positive external load $\tau_{\mathrm{load}}$ act counterclockwise. Their combined opposing torque is shown with one arrow. The rotor inertia $J$ is a property of the rotor, not an opposing torque.

</div>

Motor torque is proportional to current:

$$\tau_m=K_ti.$$

With our torque constant, $2\ \mathrm{A}$ produces $0.2\ \mathrm{N\cdot m}$.

**Inertia** describes resistance to a change in rotational motion. For the same net torque, a rotor with greater inertia accelerates more slowly. The rotational form of Newton's second law is:

$$J\frac{d\omega}{dt}=\tau_{\mathrm{net}}.$$

The derivative $d\omega/dt$ is angular acceleration, measured in rad/s². A constant acceleration of $3\ \mathrm{rad/s^2}$ maintained for two seconds increases speed by $6\ \mathrm{rad/s}$.

Our viscous friction contribution is $-b\omega$: it always opposes the direction of rotation. Including friction and the external load gives:

$$
J\frac{d\omega}{dt}=K_ti-b\omega-\tau_{\mathrm{load}}.
$$

Isolating the derivative:

$$
\boxed{\frac{d\omega}{dt}=\frac{K_ti-b\omega-\tau_{\mathrm{load}}}{J}}.
$$

The numerator is **net torque**. Positive net torque increases signed angular velocity; zero net torque means speed is momentarily constant.

At our exact starting instant, current is zero. With zero speed and no external load, acceleration is also zero. Current immediately begins increasing, producing torque and then acceleration. A positive applied voltage does not create an instantaneous jump in either current or speed.

**Unit check:** N·m divided by kg·m² gives 1/s², conventionally written rad/s² for angular acceleration.

## 1.5 Two equations, one coupled state

Our complete state and its derivative are:

$$
\mathbf{x}=\begin{bmatrix}i\\\omega\end{bmatrix},
\qquad
\frac{d\mathbf{x}}{dt}=
\begin{bmatrix}
(V-Ri-K_e\omega)/L\\
(K_ti-b\omega-\tau_{\mathrm{load}})/J
\end{bmatrix}.
$$

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Implementation note**

The current equation needs speed, and the speed equation needs current. Both derivatives must be evaluated from the **same state**. Do not update current and then use that updated current with the old speed to calculate the other derivative.

</div>

### Worked example: a snapshot during motion

Suppose $i=2\ \mathrm{A}$ and $\omega=20\ \mathrm{rad/s}$, still with a 6 V supply and no external load. This is an illustrative state, not a claim about a particular point on the startup trajectory.

The back EMF is $0.1\times20=2\ \mathrm{V}$, and the resistive drop is $1\times2=2\ \mathrm{V}$. Therefore:

$$\frac{di}{dt}=\frac{6-2-2}{0.01}=200\ \mathrm{A/s}.$$

Motor torque is $0.1\times2=0.2\ \mathrm{N\cdot m}$, while friction contributes an opposing $0.001\times20=0.02\ \mathrm{N\cdot m}$. Therefore:

$$\frac{d\omega}{dt}=\frac{0.2-0.02}{0.01}=18\ \mathrm{rad/s^2}.$$

Current and speed are both increasing at this snapshot. These two numbers are slopes, not the next state.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 2 — Return the state derivatives

Write a function named `motor_dynamics` that accepts time `t`, state `state`, applied `voltage`, and `load_torque`. Use the motor parameters defined above. For now, those parameters can remain notebook variables.

Unpack the state as `current, omega`. Calculate back EMF, motor torque, the current derivative, and the speed derivative. Return **`[di_dt, domega_dt]`**. The solver needs a time argument even though our constant-input equations do not explicitly use it.

The function must not advance time or modify the supplied state. It describes how fast the state is changing; integration is a separate job.

**Completion checks:**

| State | Current derivative | Speed derivative |
| --- | --- | --- |
| `[0, 0]` | 600 A/s | 0 rad/s² |
| `[2, 20]` | 200 A/s | 18 rad/s² |

Use 6 V and zero external load for both checks. If either fails, check signs and state order before continuing.

</div>


<details>
<summary>Show a hint</summary>

Calculate both derivatives from the original current and speed. Return the slopes, without multiplying by a time step.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
def motor_dynamics(t, state, voltage, load_torque):
    current, omega = state
    back_emf = K_e * omega
    motor_torque = K_t * current
    di_dt = (voltage - R * current - back_emf) / L
    domega_dt = (motor_torque - b * omega - load_torque) / J
    return [di_dt, domega_dt]

np.testing.assert_allclose(
    motor_dynamics(0.0, [0.0, 0.0], voltage, load_torque),
    [600.0, 0.0],
)
np.testing.assert_allclose(
    motor_dynamics(0.0, [2.0, 20.0], voltage, load_torque),
    [200.0, 18.0],
)
```

The function uses the parameters from Exercise 1 and leaves the input state unchanged. Its time argument is required by the solver even though these equations have no explicit time dependence. The two assertions check the hand-calculated examples before integration.

</details>


## 1.6 From slopes to a trajectory

A derivative tells us how fast something changes. **Integration** accumulates those changes to obtain the value over time. For current:

$$i(t)=i(0)+\int_0^t\frac{di}{dt}(s)\,ds.$$

Here $s$ is just a time variable inside the integral. Geometrically, the integral is the signed area under the current-slope curve.

Over a sufficiently short interval, we can approximate a changing slope as constant. This gives the forward Euler approximation:

$$
\begin{aligned}
i_{n+1}&\approx i_n+\Delta t\left(\frac{di}{dt}\right)_n,\\
\omega_{n+1}&\approx\omega_n+\Delta t\left(\frac{d\omega}{dt}\right)_n.
\end{aligned}
$$

The subscript $n$ labels a sample; $\Delta t$ is elapsed time. Starting from the worked snapshot above with $\Delta t=0.001\ \mathrm{s}$:

$$
\begin{aligned}
i_{n+1}&\approx2+0.001(200)=2.2\ \mathrm{A},\\
\omega_{n+1}&\approx20+0.001(18)=20.018\ \mathrm{rad/s}.
\end{aligned}
$$

Both slopes come from the original state. At the new state, the slopes change, so they must be recalculated. Euler's method becomes inaccurate—and can become unstable—if the step is too large. This hand calculation explains integration; it is not the solver we will use for the full experiment.

### Use an adaptive solver for the experiment

SciPy's `solve_ivp` solves an **initial value problem**: given starting values and a function that returns derivatives, estimate the trajectory over a time interval. Its default method uses multiple slope evaluations and adjusts internal step sizes to control local numerical error.

The winding's electrical timescale is:

$$\tau_e=\frac{L}{R}=0.01\ \mathrm{s}.$$

This is the current-response time constant when speed is held fixed. Our complete motor is coupled, so it is not the settling time of the whole system. It tells us that the first few hundredths of a second can contain important electrical behavior.

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Key distinction**

**Output sampling is different from integration steps.** The `t_eval` argument specifies times at which you want results. It does not force the solver to use those times as its internal steps. A dense output grid makes plots easier to read; numerical tolerances control integration accuracy.

</div>

### Python tools used in this exercise

Import NumPy as `np`, and import `solve_ivp` from `scipy.integrate`. `np.linspace(start, stop, count)` creates an evenly spaced array including both endpoints. A solver callback is simply a function passed to another function: SciPy calls your derivative function as it integrates. Use `args=(voltage, load_torque)` to supply its extra inputs.

The result's `y` array has one row per state variable and one column per saved time. Thus `result.y[0]` is the current history, `result.y[1]` the speed history, and `result.y[:, -1]` the final state. The colon selects all rows; `-1` selects the last column. `np.testing.assert_allclose(actual, expected, ...)` checks approximate equality and raises an error if the condition fails. An assertion is a diagnostic check, not part of the motor physics.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 3 — Simulate startup

Use `scipy.integrate.solve_ivp` to integrate from 0 to 8 seconds with initial state `[0, 0]`, a constant 6 V input, and zero external load.

- Connect the derivative function using the solver's `args` parameter or a wrapper that accepts `(t, state)`.
- Use `numpy.linspace` to request **8,001 output samples**, including both endpoints.
- Use `rtol=1e-7` and `atol=1e-9` for this small example. Relative tolerance scales with the state magnitude; absolute tolerance matters near zero. Neither is a promise of exact global error.
- Check the returned `success` flag before using the results; read `message` if it is false.
- Extract time from `t`, current from row 0 of `y`, and angular velocity from row 1 of `y`.
- Calculate motor torque from current and create a constant voltage array for plotting.

**Completion check:** time, current, speed, torque, and voltage each contain 8,001 samples; the first state is zero and all results are finite. The interval between requested output samples is 0.001 s.

</div>


<details>
<summary>Show a hint</summary>

Pass the two inputs through args. The solver result stores each state variable in a row of y, with time samples along the columns.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
from scipy.integrate import solve_ivp

time_samples = np.linspace(0.0, 8.0, 8001)
startup = solve_ivp(
    motor_dynamics,
    t_span=(0.0, 8.0),
    y0=initial_state,
    args=(voltage, load_torque),
    t_eval=time_samples,
    rtol=1e-7,
    atol=1e-9,
)
if not startup.success:
    raise RuntimeError(startup.message)

time = startup.t
current, omega = startup.y
motor_torque = K_t * current
applied_voltage = np.full_like(time, voltage)

for values in (time, current, omega, motor_torque, applied_voltage):
    assert values.shape == (8001,)
    assert np.all(np.isfinite(values))
np.testing.assert_allclose(startup.y[:, 0], initial_state)
np.testing.assert_allclose(np.diff(time), 0.001)
```

The args tuple supplies voltage and load after the solver-provided time and state. The requested grid contains both endpoints, so 8,001 samples produce 8,000 intervals. Torque is calculated from current rather than added as a redundant state variable.

</details>


## 1.7 Understand the startup plots

For the parameters in this lesson, the expected story is:

1. **At connection:** the supply is already 6 V at the start of our simulated interval. Current is zero but has a positive slope. Speed is zero.
2. **Early startup:** current rises and creates torque. The rotor begins accelerating. Because speed is initially low, back EMF is initially small.
3. **As speed increases:** back EMF increases, reducing the voltage across the inductance. Current reaches a peak and then declines. Motor torque follows the same shape because $\tau_m=K_ti$.
4. **Near steady state:** motor torque approaches the friction torque. Net torque and acceleration approach zero, while speed remains positive.

The current peak occurs when $di/dt=0$, not when current is zero. Likewise, steady speed means $d\omega/dt=0$, not $\omega=0$.

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Key distinction**

Steady speed means balanced torque—not zero motor torque or zero current. We call this experiment “unloaded” because external load torque is zero. It still includes friction, so steady-state current is nonzero: the motor must produce torque to balance that friction.

</div>

These expected shapes belong to this parameter set and experiment. Other motor parameters or inputs can produce different transients.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 4 — Make the behavior visible

Create four vertically stacked Matplotlib panels sharing a time axis:

1. Applied voltage, in V.
2. Current, in A.
3. Motor torque, in N·m.
4. Angular velocity, in rad/s.

Label axes and add a separate current close-up covering the first **0.1 seconds**. An eight-second plot can make a fast continuous rise look like an instantaneous jump.

**Completion check:** torque is current multiplied by `K_t`, current rises from zero before declining, and speed approaches a positive plateau. If you write observations beneath the plots, describe the results you actually obtained rather than copying the expected behavior.

</div>


<details>
<summary>Show a hint</summary>

Use subplots with sharex=True for the four panels. Make a separate figure with xlim from 0 to 0.1 seconds for the startup detail.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 1, sharex=True, figsize=(10, 9), layout="constrained")
series = [
    (applied_voltage, "Voltage (V)"),
    (current, "Current (A)"),
    (motor_torque, "Torque (N m)"),
    (omega, "Speed (rad/s)"),
]
for ax, (values, label) in zip(axes, series):
    ax.plot(time, values)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("DC motor startup: 6 V, zero external load")

fig_detail, ax_detail = plt.subplots(figsize=(10, 3), layout="constrained")
ax_detail.plot(time, current)
ax_detail.set(
    xlim=(0.0, 0.1),
    xlabel="Time (s)",
    ylabel="Current (A)",
    title="Current during the first 0.1 seconds",
)
ax_detail.grid(True, alpha=0.3)
plt.show()
```

The shared time axis aligns electrical and mechanical behavior. The separate close-up reveals the continuous current rise that is compressed in the full-duration plot. Torque has the same shape as current because their relationship is a constant scale factor.

</details>


## 1.8 Predict the final values independently

At steady state, both state derivatives are zero. For a constant voltage and zero external load:

$$
\begin{aligned}
V&=Ri_{\mathrm{ss}}+K_e\omega_{\mathrm{ss}},\\
K_ti_{\mathrm{ss}}&=b\omega_{\mathrm{ss}}.
\end{aligned}
$$

The subscript $\mathrm{ss}$ means steady state. The second equation expresses torque balance. Solve it for current:

$$i_{\mathrm{ss}}=\frac{b\omega_{\mathrm{ss}}}{K_t}.$$

Substitute into the voltage balance:

$$
V=R\frac{b\omega_{\mathrm{ss}}}{K_t}+K_e\omega_{\mathrm{ss}}
=\left(\frac{Rb}{K_t}+K_e\right)\omega_{\mathrm{ss}}.
$$

Rearranging gives:

$$
\boxed{\omega_{\mathrm{ss}}=\frac{K_tV}{Rb+K_tK_e}},
\qquad
\boxed{i_{\mathrm{ss}}=\frac{b\omega_{\mathrm{ss}}}{K_t}}.
$$

For our parameters:

$$
\omega_{\mathrm{ss}}=\frac{0.1(6)}{1(0.001)+0.1(0.1)}
\approx54.545\ \mathrm{rad/s}.
$$

| Quantity | Predicted steady-state value |
| --- | --- |
| Angular velocity | 54.545 rad/s, about 521 rpm |
| Current | 0.54545 A |
| Motor torque | 0.054545 N·m |
| Back EMF | 5.4545 V |
| Resistive voltage drop | 0.54545 V |

Back EMF plus the resistive drop accounts for the 6 V supply. Motor torque balances friction. Notice that $L$ and $J$ disappear from these steady-state equations: they affect how the motor gets there, but not this equilibrium.

### Check a finite simulation fairly

Steady state is a limiting condition. A sample at eight seconds need not equal the limiting value exactly. For this experiment, final current and speed should each be within **1%** of the predicted values.

Relative error can be calculated as:

$$\text{relative error}=\frac{|\text{simulated}-\text{predicted}|}{|\text{predicted}|}.$$

If the trajectory is still approaching the prediction, extend the simulation. If it has settled at a different value, check the model and units. If tightening tolerances materially changes the result, investigate numerical accuracy.

A particularly useful check is to evaluate your derivative function at the analytically calculated steady state. Both derivatives should be zero to floating-point precision, independently of how long you integrate.

| Symptom | First thing to inspect |
| --- | --- |
| Startup derivative check fails | Parameter values, signs, and state order |
| Speed runs away | Back EMF and friction should oppose positive motoring |
| Current or speed has an implausible scale | Units, especially rpm versus rad/s |
| Torque plot has a different shape from current | Torque must be calculated as $K_ti$ |
| Final result misses equilibrium but is still changing | Simulation duration |
| Zero input creates motion from rest | State initialization, load, and derivative equations |

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 5 — Validate the model

Calculate the analytical steady-state speed and current from the parameters, rather than hard-coding the rounded table values. Then:

1. Compare them with the last simulation samples. Report the relative errors and check that each is below 1%.
2. Pass the analytical steady-state current and speed into `motor_dynamics`. Both returned derivatives should be approximately zero; an absolute tolerance of `1e-8` is sufficient for these values.
3. Run a separate experiment with zero applied voltage, zero external load, and initial state `[0, 0]`. Both states should remain zero within numerical tolerance.

Keep your startup results available for the earlier plots. Passing these checks gives us evidence for the model's implementation; it does not establish that these illustrative parameters describe a particular physical motor.

</div>


<details>
<summary>Show a hint</summary>

Set both derivatives to zero to calculate equilibrium. Keep the zero-voltage result in a separate variable so the startup arrays remain available.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
steady_speed = K_t * voltage / (R * b + K_t * K_e)
steady_current = b * steady_speed / K_t

speed_error = abs(omega[-1] - steady_speed) / abs(steady_speed)
current_error = abs(current[-1] - steady_current) / abs(steady_current)
print(f"Final speed: {omega[-1]:.5f} rad/s; relative error: {speed_error:.3%}")
print(f"Final current: {current[-1]:.5f} A; relative error: {current_error:.3%}")
assert speed_error < 0.01
assert current_error < 0.01

np.testing.assert_allclose(
    motor_dynamics(0.0, [steady_current, steady_speed], voltage, 0.0),
    [0.0, 0.0],
    rtol=0.0,
    atol=1e-8,
)

zero_input = solve_ivp(
    motor_dynamics,
    t_span=(0.0, 8.0),
    y0=[0.0, 0.0],
    args=(0.0, 0.0),
    t_eval=time_samples,
    rtol=1e-7,
    atol=1e-9,
)
if not zero_input.success:
    raise RuntimeError(zero_input.message)
np.testing.assert_allclose(zero_input.y, 0.0, rtol=0.0, atol=1e-9)
```

The equilibrium formulas apply to this zero-load experiment. Using their unrounded values avoids artificial errors in the derivative check. After eight seconds, the expected relative errors are about 0.014% for speed and 0.141% for current. The zero-input experiment checks that the model does not create motion from rest without an input.

</details>


## What you can now build on

The motor is described by two coupled rates of change:

$$
\frac{di}{dt}=\frac{V-Ri-K_e\omega}{L},
\qquad
\frac{d\omega}{dt}=\frac{K_ti-b\omega-\tau_{\mathrm{load}}}{J}.
$$

Current creates torque. Net torque changes speed. Speed creates back EMF, which changes current. The solver integrates those interacting changes over time.

Once your exercises pass, you have a reusable derivative function, a startup trajectory, plots explaining the transient, and independent equilibrium checks. In Section 2, we'll reuse this model to see how fixed voltages and different loads affect speed—and why a fixed voltage alone cannot maintain a desired speed as conditions change.

[Next lesson: Open-Loop Control](02_open_loop_control.ipynb)

## References and further reading

[SciPy solve_ivp reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) — callback arguments, output arrays, sample times, and tolerances.